# Hospital Analytics ETL

This notebook loads the raw hospital patient-flow dataset into a pandas DataFrame for future ETL work. No cleaning or transformation is performed at this stage.

In [1]:
import pandas as pd

df = pd.read_csv("../01_Dataset/healthcare_analytics_patient_flow_data.csv")
df.head()

,Patient Id,Patient Admission Date,Patient Admission Time,Merged,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,780-96-6113,9/9/2024,9:25:00 AM,W. Breede,Female,63,African American,NaN,Not Admission,5.0,32
1,714-35-6722,9/9/2024,4:42:00 PM,Y. Baldetti,Male,31,Asian,Orthopedics,Not Admission,NaN,22
2,571-85-3714,9/9/2024,12:14:00 AM,M. Semerad,Male,75,White,General Practice,Not Admission,NaN,16
3,404-43-9499,9/9/2024,8:33:00 PM,K. Blaydes,Male,79,African American,General Practice,Admission,NaN,38
4,552-51-5855,9/9/2024,7:25:00 PM,F. Dickerson,Female,24,African American,NaN,Admission,NaN,36


In [2]:
df.shape

(9216, 11)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9216 entries, 0 to 9215
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient Id                  9216 non-null   object 
 1   Patient Admission Date      9216 non-null   object 
 2   Patient Admission Time      9216 non-null   object 
 3   Merged                      9216 non-null   object 
 4   Patient Gender              9216 non-null   object 
 5   Patient Age                 9216 non-null   int64  
 6   Patient Race                9216 non-null   object 
 7   Department Referral         3816 non-null   object 
 8   Patient Admission Flag      9216 non-null   object 
 9   Patient Satisfaction Score  2517 non-null   float64
 10  Patient Waittime            9216 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 792.1+ KB


In [4]:
# Explicit missing-value check
df.isna().sum()

Patient Id                       0
Patient Admission Date           0
Patient Admission Time           0
Merged                           0
Patient Gender                   0
Patient Age                      0
Patient Race                     0
Department Referral           5400
Patient Admission Flag           0
Patient Satisfaction Score    6699
Patient Waittime                 0
dtype: int64

In [5]:
# Calculate the percentages in Python
(df.isna().sum() / len(df) * 100).round(2)

Patient Id                     0.00
Patient Admission Date         0.00
Patient Admission Time         0.00
Merged                         0.00
Patient Gender                 0.00
Patient Age                    0.00
Patient Race                   0.00
Department Referral           58.59
Patient Admission Flag         0.00
Patient Satisfaction Score    72.69
Patient Waittime               0.00
dtype: float64

In [6]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [7]:
# Check Patient ID uniqueness
df["Patient Id"].nunique()

9216

In [8]:
# Check categorical values
df["Patient Gender"].value_counts(dropna=False)

Patient Gender
Male           4729
Female         4470
Femaleemale      17
Name: count, dtype: int64

In [9]:
# Check Department Referral
df["Department Referral"].value_counts(dropna=False)

Department Referral
NaN                 5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [10]:
# Check Admission Flag
df["Patient Admission Flag"].value_counts(dropna=False)

Patient Admission Flag
Admission        4612
Not Admission    4604
Name: count, dtype: int64

In [11]:
# Inspect Satisfaction Score
df["Patient Satisfaction Score"].value_counts(dropna=False).sort_index()

Patient Satisfaction Score
0.0      222
1.0      246
2.0      204
3.0      228
4.0      248
5.0      221
6.0      231
7.0      256
8.0      218
9.0      222
10.0     221
NaN     6699
Name: count, dtype: int64

In [12]:
# Validate Age and Wait Time ranges
df[["Patient Age", "Patient Waittime"]].describe()

,Patient Age,Patient Waittime
count,9216.000000,9216.000000
mean,39.855143,35.259874
std,22.755125,14.735323
min,1.000000,10.000000
25%,20.000000,23.000000
50%,39.000000,35.000000
75%,60.000000,48.000000
max,79.000000,60.000000


In [13]:
# Validate the Admission Date
df["Patient Admission Date"].head(10)

0    9/9/2024
1    9/9/2024
2    9/9/2024
3    9/9/2024
4    9/9/2024
5    9/9/2024
6    9/9/2024
7    9/9/2024
8    9/9/2024
9    9/9/2024
Name: Patient Admission Date, dtype: object

In [14]:
# Test date conversion without modifying the data
test_dates = pd.to_datetime(
    df["Patient Admission Date"],
    dayfirst=True,
    errors="coerce"
)

test_dates.isna().sum()

np.int64(0)

In [15]:
# Check the converted date range
print("Minimum Date:", test_dates.min())
print("Maximum Date:", test_dates.max())

Minimum Date: 2023-04-01 00:00:00
Maximum Date: 2024-10-30 00:00:00


In [16]:
#determine the real date format
df["Patient Admission Date"].tail(20)

9196    1/10/2023
9197     1/1/2024
9198     1/1/2024
9199     1/1/2024
9200     1/1/2024
9201     1/1/2024
9202     1/1/2024
9203     1/1/2024
9204     1/1/2024
9205     1/1/2024
9206     1/1/2024
9207     1/1/2024
9208     1/1/2024
9209     1/1/2024
9210     1/1/2024
9211     1/1/2024
9212     1/1/2024
9213     1/1/2024
9214     1/1/2024
9215     1/1/2024
Name: Patient Admission Date, dtype: object

In [17]:
# Find dates with a component greater than 12
df[
    df["Patient Admission Date"]
    .str.split("/")
    .str[0]
    .astype(int) > 12
]["Patient Admission Date"].head(20)

1739    31/12/2023
1740    31/12/2023
1741    31/12/2023
1742    31/12/2023
1743    31/12/2023
1744    31/12/2023
1745    31/12/2023
1746    31/12/2023
1747    31/12/2023
1748    31/12/2023
1749    31/12/2023
1750    31/12/2023
1751    31/12/2023
1752    31/12/2023
1753    31/12/2023
1754    31/10/2023
1755    31/10/2023
1756    31/10/2023
1757    31/10/2023
1758    31/10/2023
Name: Patient Admission Date, dtype: object

In [18]:
# Validate Admission Time
test_times = pd.to_datetime(
    df["Patient Admission Time"],
    format="%I:%M:%S %p",
    errors="coerce"
)

test_times.isna().sum()

np.int64(0)

# start the actual ETL

In [19]:
# Create a working copy for ETL
hospital_clean = df.copy()
hospital_clean.shape

(9216, 11)

In [20]:
# Drop the Merged column
hospital_clean.columns

Index(['Patient Id', 'Patient Admission Date', 'Patient Admission Time',
       'Merged', 'Patient Gender', 'Patient Age', 'Patient Race',
       'Department Referral', 'Patient Admission Flag',
       'Patient Satisfaction Score', 'Patient Waittime'],
      dtype='object')

In [21]:
# Remove corrupted/unnecessary patient name field
hospital_clean = hospital_clean.drop(columns=["Merged"])

In [22]:
hospital_clean.shape

(9216, 10)

In [23]:
hospital_clean.columns

Index(['Patient Id', 'Patient Admission Date', 'Patient Admission Time',
       'Patient Gender', 'Patient Age', 'Patient Race', 'Department Referral',
       'Patient Admission Flag', 'Patient Satisfaction Score',
       'Patient Waittime'],
      dtype='object')

### Clean Patient Gender

In [24]:
# Check Patient Gender before cleaning
hospital_clean["Patient Gender"].value_counts(dropna=False)

Patient Gender
Male           4729
Female         4470
Femaleemale      17
Name: count, dtype: int64

In [25]:
# Correct invalid gender value
hospital_clean["Patient Gender"] = hospital_clean["Patient Gender"].replace(
    "Femaleemale", "Female"
)

In [26]:
# Validate Patient Gender after cleaning
hospital_clean["Patient Gender"].value_counts(dropna=False)

Patient Gender
Male      4729
Female    4487
Name: count, dtype: int64

### Clean Department Referral

In [27]:
# Check Department Referral before cleaning
hospital_clean["Department Referral"].value_counts(dropna=False)

Department Referral
NaN                 5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [28]:
# Replace missing values with "None"
# Preserve missing referral as the source dataset category "None"
hospital_clean["Department Referral"] = (
    hospital_clean["Department Referral"].fillna("None")
)

In [29]:
# Validate Department Referral after cleaning
hospital_clean["Department Referral"].value_counts(dropna=False)

Department Referral
None                5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [30]:
hospital_clean["Department Referral"].isna().sum()

np.int64(0)

### Convert Patient Admission Date

In [31]:
# Check Admission Date datatype before conversion
hospital_clean["Patient Admission Date"].dtype

dtype('O')

In [32]:
# Convert Patient Admission Date to datetime
hospital_clean["Patient Admission Date"] = pd.to_datetime(
    hospital_clean["Patient Admission Date"],
    dayfirst=True,
    errors="coerce"
)

In [33]:
# Check datatype after conversion
hospital_clean["Patient Admission Date"].dtype

dtype('<M8[ns]')

In [34]:
# Check for invalid/missing dates after conversion
hospital_clean["Patient Admission Date"].isna().sum()

np.int64(0)

In [35]:
# Check converted dates
hospital_clean["Patient Admission Date"].head()

0   2024-09-09
1   2024-09-09
2   2024-09-09
3   2024-09-09
4   2024-09-09
Name: Patient Admission Date, dtype: datetime64[ns]

### Convert Patient Admission Time

In [36]:
# Check current datatype
hospital_clean["Patient Admission Time"].dtype

dtype('O')

In [37]:
# Convert the time values
hospital_clean["Patient Admission Time"] = pd.to_datetime(
    hospital_clean["Patient Admission Time"],
    format="%I:%M:%S %p",
    errors="coerce"
)

In [38]:
# Check datatype after conversion
hospital_clean["Patient Admission Time"].dtype

dtype('<M8[ns]')

In [39]:
# Check for invalid/missing time after conversion
hospital_clean["Patient Admission Time"].isna().sum()

np.int64(0)

In [40]:
# Check converted time
hospital_clean["Patient Admission Time"].head()

0   1900-01-01 09:25:00
1   1900-01-01 16:42:00
2   1900-01-01 00:14:00
3   1900-01-01 20:33:00
4   1900-01-01 19:25:00
Name: Patient Admission Time, dtype: datetime64[ns]

In [41]:
# Keep only the time component
hospital_clean["Patient Admission Time"] = (
    hospital_clean["Patient Admission Time"].dt.time
)

In [42]:
# check
hospital_clean["Patient Admission Time"].head()

0    09:25:00
1    16:42:00
2    00:14:00
3    20:33:00
4    19:25:00
Name: Patient Admission Time, dtype: object

In [43]:
# Check datatype after conversion
hospital_clean["Patient Admission Time"].dtype

dtype('O')

In [44]:
# Check remaining missing values after cleaning so far
hospital_clean.isna().sum()

Patient Id                       0
Patient Admission Date           0
Patient Admission Time           0
Patient Gender                   0
Patient Age                      0
Patient Race                     0
Department Referral              0
Patient Admission Flag           0
Patient Satisfaction Score    6699
Patient Waittime                 0
dtype: int64

In [45]:
# Check Satisfaction Score availability by Admission Flag
pd.crosstab(
    hospital_clean["Patient Admission Flag"],
    hospital_clean["Patient Satisfaction Score"].isna(),
    margins=True
)

Patient Satisfaction Score,False,True,All
Patient Admission Flag,,,
Admission,1235,3377,4612
Not Admission,1282,3322,4604
All,2517,6699,9216


In [46]:
# Validate numeric columns
hospital_clean[
    ["Patient Age", "Patient Waittime", "Patient Satisfaction Score"]
].describe()

,Patient Age,Patient Waittime,Patient Satisfaction Score
count,9216.000000,9216.000000,2517.000000
mean,39.855143,35.259874,4.992054
std,22.755125,14.735323,3.138043
min,1.000000,10.000000,0.000000
25%,20.000000,23.000000,2.000000
50%,39.000000,35.000000,5.000000
75%,60.000000,48.000000,8.000000
max,79.000000,60.000000,10.000000


In [47]:
# Create Age Group for analysis
hospital_clean["Age Group"] = pd.cut(
    hospital_clean["Patient Age"],
    bins=[0, 17, 34, 49, 64, float("inf")],
    labels=["Child", "Young Adult", "Adult", "Older Adult", "Senior"]
)

In [48]:
# validate
hospital_clean["Age Group"].value_counts().sort_index()

Age Group
Child          1971
Young Adult    2019
Adult          1768
Older Adult    1722
Senior         1736
Name: count, dtype: int64

In [49]:
hospital_clean["Age Group"].isna().sum()

np.int64(0)

In [50]:
hospital_clean[["Patient Age", "Age Group"]].head(10)

,Patient Age,Age Group
0,63,Older Adult
1,31,Young Adult
2,75,Senior
3,79,Senior
4,24,Young Adult
5,27,Young Adult
6,70,Senior
7,64,Older Adult
8,69,Senior
9,25,Young Adult


In [51]:
# Create Admission Hour
hospital_clean["Admission Hour"] = hospital_clean[
    "Patient Admission Time"
].apply(lambda x: x.hour)

In [52]:
# Validate Admission Hour
hospital_clean["Admission Hour"].value_counts().sort_index()

Admission Hour
0     406
1     372
2     376
3     385
4     384
5     393
6     375
7     415
8     386
9     388
10    349
11    403
12    366
13    410
14    368
15    394
16    378
17    359
18    370
19    383
20    372
21    376
22    372
23    436
Name: count, dtype: int64

In [53]:
hospital_clean["Admission Hour"].isna().sum()

np.int64(0)

In [54]:
hospital_clean[
    ["Patient Admission Time", "Admission Hour"]
].head(10)

,Patient Admission Time,Admission Hour
0,09:25:00,9
1,16:42:00,16
2,00:14:00,0
3,20:33:00,20
4,19:25:00,19
5,12:05:00,12
6,17:25:00,17
7,19:46:00,19
8,12:34:00,12
9,13:14:00,13


In [55]:
# Create Time of Day
hospital_clean["Time of Day"] = pd.cut(
    hospital_clean["Admission Hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"]
)

In [56]:
# Validate Time of Day
hospital_clean["Time of Day"].value_counts().sort_index()

Time of Day
Night        2316
Morning      2316
Afternoon    2275
Evening      2309
Name: count, dtype: int64

In [57]:
hospital_clean["Time of Day"].isna().sum()

np.int64(0)

In [58]:
hospital_clean[
    ["Patient Admission Time", "Admission Hour", "Time of Day"]
].head(10)

,Patient Admission Time,Admission Hour,Time of Day
0,09:25:00,9,Morning
1,16:42:00,16,Afternoon
2,00:14:00,0,Night
3,20:33:00,20,Evening
4,19:25:00,19,Evening
5,12:05:00,12,Afternoon
6,17:25:00,17,Afternoon
7,19:46:00,19,Evening
8,12:34:00,12,Afternoon
9,13:14:00,13,Afternoon


### Create Day-of-Week Features

In [59]:
# Create Admission Day
hospital_clean["Admission Day"] = (
    hospital_clean["Patient Admission Date"].dt.day_name()
)

In [60]:
# Validate Admission Day
hospital_clean["Admission Day"].value_counts()

Admission Day
Saturday     1377
Thursday     1332
Sunday       1318
Monday       1314
Friday       1310
Tuesday      1305
Wednesday    1260
Name: count, dtype: int64

In [61]:
# Create Weekend Flag
hospital_clean["Day Type"] = hospital_clean[
    "Patient Admission Date"
].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

In [62]:
hospital_clean["Day Type"].value_counts()

Day Type
Weekday    6521
Weekend    2695
Name: count, dtype: int64

In [63]:
hospital_clean[
    ["Patient Admission Date", "Admission Day", "Day Type"]
].head(10)

,Patient Admission Date,Admission Day,Day Type
0,2024-09-09,Monday,Weekday
1,2024-09-09,Monday,Weekday
2,2024-09-09,Monday,Weekday
3,2024-09-09,Monday,Weekday
4,2024-09-09,Monday,Weekday
5,2024-09-09,Monday,Weekday
6,2024-09-09,Monday,Weekday
7,2024-09-09,Monday,Weekday
8,2024-09-09,Monday,Weekday
9,2024-09-09,Monday,Weekday


In [64]:
# Check missing values in derived day features
hospital_clean[["Admission Day", "Day Type"]].isna().sum()

Admission Day    0
Day Type         0
dtype: int64

### Final Data Validation

In [65]:
# Check final shape and columns
print("Shape:", hospital_clean.shape)

hospital_clean.info()

Shape: (9216, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9216 entries, 0 to 9215
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Patient Id                  9216 non-null   object        
 1   Patient Admission Date      9216 non-null   datetime64[ns]
 2   Patient Admission Time      9216 non-null   object        
 3   Patient Gender              9216 non-null   object        
 4   Patient Age                 9216 non-null   int64         
 5   Patient Race                9216 non-null   object        
 6   Department Referral         9216 non-null   object        
 7   Patient Admission Flag      9216 non-null   object        
 8   Patient Satisfaction Score  2517 non-null   float64       
 9   Patient Waittime            9216 non-null   int64         
 10  Age Group                   9216 non-null   category      
 11  Admission Hour              9216 non-n

In [66]:
# Final missing-value check
hospital_clean.isna().sum()

Patient Id                       0
Patient Admission Date           0
Patient Admission Time           0
Patient Gender                   0
Patient Age                      0
Patient Race                     0
Department Referral              0
Patient Admission Flag           0
Patient Satisfaction Score    6699
Patient Waittime                 0
Age Group                        0
Admission Hour                   0
Time of Day                      0
Admission Day                    0
Day Type                         0
dtype: int64

In [67]:
# Check duplicate Patient IDs
hospital_clean["Patient Id"].duplicated().sum()

np.int64(0)

In [68]:
# Final validation of cleaned dataset

print("Rows:", len(hospital_clean))
print("Columns:", len(hospital_clean.columns))
print("Duplicate rows:", hospital_clean.duplicated().sum())
print("Duplicate Patient IDs:", hospital_clean["Patient Id"].duplicated().sum())

hospital_clean.head()

Rows: 9216
Columns: 15
Duplicate rows: 0
Duplicate Patient IDs: 0


,Patient Id,Patient Admission Date,Patient Admission Time,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Age Group,Admission Hour,Time of Day,Admission Day,Day Type
0,780-96-6113,2024-09-09,09:25:00,Female,63,African American,None,Not Admission,5.0,32,Older Adult,9,Morning,Monday,Weekday
1,714-35-6722,2024-09-09,16:42:00,Male,31,Asian,Orthopedics,Not Admission,NaN,22,Young Adult,16,Afternoon,Monday,Weekday
2,571-85-3714,2024-09-09,00:14:00,Male,75,White,General Practice,Not Admission,NaN,16,Senior,0,Night,Monday,Weekday
3,404-43-9499,2024-09-09,20:33:00,Male,79,African American,General Practice,Admission,NaN,38,Senior,20,Evening,Monday,Weekday
4,552-51-5855,2024-09-09,19:25:00,Female,24,African American,None,Admission,NaN,36,Young Adult,19,Evening,Monday,Weekday


In [69]:
# Export cleaned hospital dataset

output_path = "../03_Output/hospital_patient_flow_clean.csv"

hospital_clean.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


In [70]:
# Verify exported CSV

check_export = pd.read_csv("../03_Output/hospital_patient_flow_clean.csv")

print("Exported shape:", check_export.shape)
check_export.head()

Exported shape: (9216, 15)


,Patient Id,Patient Admission Date,Patient Admission Time,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Age Group,Admission Hour,Time of Day,Admission Day,Day Type
0,780-96-6113,2024-09-09,09:25:00,Female,63,African American,NaN,Not Admission,5.0,32,Older Adult,9,Morning,Monday,Weekday
1,714-35-6722,2024-09-09,16:42:00,Male,31,Asian,Orthopedics,Not Admission,NaN,22,Young Adult,16,Afternoon,Monday,Weekday
2,571-85-3714,2024-09-09,00:14:00,Male,75,White,General Practice,Not Admission,NaN,16,Senior,0,Night,Monday,Weekday
3,404-43-9499,2024-09-09,20:33:00,Male,79,African American,General Practice,Admission,NaN,38,Senior,20,Evening,Monday,Weekday
4,552-51-5855,2024-09-09,19:25:00,Female,24,African American,NaN,Admission,NaN,36,Young Adult,19,Evening,Monday,Weekday


In [71]:
# Verify Department Referral after CSV reload

check_export["Department Referral"].value_counts(dropna=False)

Department Referral
NaN                 5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [72]:
hospital_clean["Department Referral"].isna().sum()

np.int64(0)

In [73]:
# Check Department Referral in cleaned DataFrame
hospital_clean["Department Referral"].value_counts(dropna=False)


Department Referral
None                5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [74]:
# Replace ambiguous "None" category with "Not Specified"
hospital_clean["Department Referral"] = (
    hospital_clean["Department Referral"]
    .replace("None", "Not Specified")
)

# Validate
hospital_clean["Department Referral"].value_counts(dropna=False)

Department Referral
Not Specified       5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [75]:
# Re-export corrected cleaned dataset
hospital_clean.to_csv(
    "../03_Output/hospital_patient_flow_clean.csv",
    index=False
)

print("Cleaned dataset re-exported successfully.")


Cleaned dataset re-exported successfully.


In [76]:
# Reload the newly exported CSV
check_export = pd.read_csv(
    "../03_Output/hospital_patient_flow_clean.csv"
)

# Check Department Referral
check_export["Department Referral"].value_counts(dropna=False)

Department Referral
Not Specified       5400
General Practice    1840
Orthopedics          995
Physiotherapy        276
Cardiology           248
Neurology            193
Gastroenterology     178
Renal                 86
Name: count, dtype: int64

In [77]:
check_export.isna().sum()

Patient Id                       0
Patient Admission Date           0
Patient Admission Time           0
Patient Gender                   0
Patient Age                      0
Patient Race                     0
Department Referral              0
Patient Admission Flag           0
Patient Satisfaction Score    6699
Patient Waittime                 0
Age Group                        0
Admission Hour                   0
Time of Day                      0
Admission Day                    0
Day Type                         0
dtype: int64

In [78]:
check_export["Department Referral"].isna().sum()

np.int64(0)

In [79]:
print("Exported shape:", check_export.shape)

Exported shape: (9216, 15)


In [80]:
check_export.dtypes

Patient Id                     object
Patient Admission Date         object
Patient Admission Time         object
Patient Gender                 object
Patient Age                     int64
Patient Race                   object
Department Referral            object
Patient Admission Flag         object
Patient Satisfaction Score    float64
Patient Waittime                int64
Age Group                      object
Admission Hour                  int64
Time of Day                    object
Admission Day                  object
Day Type                       object
dtype: object

In [81]:
check_export.head(3)

,Patient Id,Patient Admission Date,Patient Admission Time,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Age Group,Admission Hour,Time of Day,Admission Day,Day Type
0,780-96-6113,2024-09-09,09:25:00,Female,63,African American,Not Specified,Not Admission,5.0,32,Older Adult,9,Morning,Monday,Weekday
1,714-35-6722,2024-09-09,16:42:00,Male,31,Asian,Orthopedics,Not Admission,NaN,22,Young Adult,16,Afternoon,Monday,Weekday
2,571-85-3714,2024-09-09,00:14:00,Male,75,White,General Practice,Not Admission,NaN,16,Senior,0,Night,Monday,Weekday


In [83]:
# Correlation between Wait Time and Satisfaction Score

correlation = hospital_clean[
    ["Patient Waittime", "Patient Satisfaction Score"]
].corr()

correlation

,Patient Waittime,Patient Satisfaction Score
Patient Waittime,1.000000,-0.021183
Patient Satisfaction Score,-0.021183,1.000000
